# 📝 Cypher 기초 과제 LV1(기초): CREATE·MATCH·WHERE·RETURN·SET·REMOVE·MERGE·DELETE

> 이 단원의 새 문법을 **하나씩** 확인합니다. 노드/관계 만들기(CREATE), 관계에 속성 담기, 찾기(MATCH), 조건 걸기(WHERE: 비교·AND·OR·NOT·IS NULL), 값 돌려받기(RETURN AS), 값 고치기(SET)와 지우기(REMOVE), 결과 다듬기(DISTINCT·ORDER BY·LIMIT), 멱등 적재(MERGE), 지우기(DELETE).

## 풀이 방법
1. 맨 위 **준비 셀 → 초기화 셀 → 시드 셀**을 위에서부터 실행하세요(카페 그래프가 만들어집니다).
2. 각 문제의 **답안 셀**에 Cypher 를 `run_cypher("...")` 로 실행하는 코드를 채우고, **자가채점 셀**로 확인하세요(✅ 통과!).
3. **`CREATE` 를 쓰는 문제(1·2·13·22·23번)의 답안 셀은 한 번만 실행하세요.** 두 번 실행하면 같은 음료·관계가 두 개가 되어 채점이 계속 떨어집니다. 그렇게 됐다면 **맨 위 초기화 셀과 시드 셀부터 다시 실행**한 뒤 1번부터 한 번씩만 풀면 됩니다(`MERGE` 를 쓰는 문제는 여러 번 실행해도 안전합니다).
4. **문제 번호 순서대로 푸세요.** **14·15번은 앞에서 만든 것을 지우는 문제**, **17·18번은 그래프의 값을 고치는 문제**라, 순서를 건너뛰면 다른 문제의 채점이 떨어집니다. 같은 이유로 **앞 문제의 채점 셀을 나중에 다시 실행하면** 결과가 달라질 수 있습니다(순서대로 한 번씩 풀었다면 다시 실행할 필요가 없습니다).

- 도메인: 카페 **달빛**의 메뉴: 음료(`Drink`: `name`·`price`·`temp`), 재료(`Ingredient`), 분류(`Category`), 관계는 `IN_CATEGORY`(음료→분류)·`HAS_INGREDIENT`(음료→재료)입니다.
- **여러 개**를 돌려받는 문제는 순서가 뒤섞여 나올 수 있으니, 파이썬에서 **정렬하거나 집합**으로 다뤄 비교합니다(순서는 정해져 있지 않습니다).

화이팅!

아래 준비 셀 3개를 먼저 실행하세요.

In [ ]:
# [제공 코드] Neo4j 연결: 실행만 하세요. 반드시 "실습 전용" DB 여야 합니다(아래 실습이 그래프를 지웁니다).
# 앞으로 모든 Cypher 는 run_cypher("쿼리", 파라미터=값) 으로 실행하고, 결과는 dict 리스트로 옵니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError  # 지우기 규칙 위반 에러

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# .env 를 못 읽어도 에러 없이 기본값으로 넘어간다. 마지막 줄에 찍히는 주소를 눈으로 꼭 확인할 것
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)   # 여기까지 찍히면 준비 완료

> ⚠️ **아래 초기화 셀은 연결된 데이터베이스의 노드를 전부 지웁니다.** 지난 단원에서 브라우저로 적재한 **Movies 예제 그래프와 그때 푼 과제 결과도 함께 사라집니다.** 되돌릴 수 없으니, `.env` 가 **실습 전용 DB** 를 가리키는지 먼저 확인하세요. Movies 를 남기고 싶다면 실습용 인스턴스를 따로 하나 만들어 그 접속 정보를 `.env` 에 넣으면 됩니다(지웠더라도 day28 폴더의 `data/movies_setup.cypher` 로 다시 적재할 수 있습니다).

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요! 노드·관계를 전부 지웁니다.
# MATCH (n) 은 모든 노드, DETACH 는 붙어 있는 관계까지 함께 지우라는 뜻입니다(교안_01 마지막 절에 나옵니다).
run_cypher("MATCH (n) DETACH DELETE n")
# 확인: MATCH (n) RETURN n 은 남은 노드를 한 줄씩 돌려주므로 그 행 수가 곧 노드 개수다
print("초기화 완료. 남은 노드:", len(run_cypher("MATCH (n) RETURN n")))

In [ ]:
# [제공 코드] 카페 "달빛" 메뉴 그래프 적재: 이 셀은 실행만 하세요.
# 음료(Drink)·재료(Ingredient)·분류(Category) 노드와 분류(IN_CATEGORY)·재료포함(HAS_INGREDIENT) 관계를 만듭니다.
drinks = [
    {"name": "아메리카노",     "price": 3000, "temp": "핫",   "category": "커피",   "ingredients": ["원두", "물"]},
    {"name": "카페라떼",       "price": 3500, "temp": "핫",   "category": "커피",   "ingredients": ["원두", "우유"]},
    {"name": "아이스아메리카노", "price": 3500, "temp": "아이스", "category": "커피",   "ingredients": ["원두", "물", "얼음"]},
    {"name": "콜드브루",       "price": 4000, "temp": "아이스", "category": "커피",   "ingredients": ["원두", "물", "얼음"]},
    {"name": "녹차라떼",       "price": 4500, "temp": "핫",   "category": "티",     "ingredients": ["녹차가루", "우유"]},
    {"name": "캐모마일티",     "price": 3800, "temp": "핫",   "category": "티",     "ingredients": ["캐모마일", "물"]},
    {"name": "딸기스무디",     "price": 5000, "temp": "아이스", "category": "스무디", "ingredients": ["딸기", "우유", "얼음"]},
]

categories = {d["category"] for d in drinks}
all_ings = {i for d in drinks for i in d["ingredients"]}
for c in categories:
    run_cypher("CREATE (:Category {name: $name})", name=c)
for i in all_ings:
    run_cypher("CREATE (:Ingredient {name: $name})", name=i)
for d in drinks:
    run_cypher(
        "CREATE (:Drink {name: $name, price: $price, temp: $temp})",
        name=d["name"], price=d["price"], temp=d["temp"],
    )
    run_cypher(
        "MATCH (d:Drink {name: $name}), (c:Category {name: $category}) "
        "CREATE (d)-[:IN_CATEGORY]->(c)",
        name=d["name"], category=d["category"],
    )
    for ing in d["ingredients"]:
        run_cypher(
            "MATCH (d:Drink {name: $name}), (i:Ingredient {name: $ing}) "
            "CREATE (d)-[:HAS_INGREDIENT]->(i)",
            name=d["name"], ing=ing,
        )

print("적재한 음료 수:", len(run_cypher("MATCH (d:Drink) RETURN d")))


이 과제가 쓰는 그래프의 구조입니다. 레이블·속성과 관계의 **방향**을 먼저 확인하세요.

<img src="images/cafe-schema.png" width="820">

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 어떤 음료가 있는지 먼저 훑어봅니다.

In [ ]:
# [제공 코드] 음료 목록을 먼저 살펴봅니다
for r in run_cypher("MATCH (d:Drink) RETURN d.name AS name, d.price AS price, d.temp AS temp"):
    print(r['name'], r['price'], r['temp'])

## 1. 새 음료 노드 만들기 (CREATE)
**배경**: 신메뉴 **바닐라라떼**가 나왔습니다. 그래프에 음료 노드를 하나 추가합니다.

**요구사항**:
- 레이블 `Drink`, 속성 `name='바닐라라떼'`, `price=4200`, `temp='핫'` 인 노드를 `CREATE` 하세요.
- 문자열 값은 작은따옴표로, 숫자(`price`)는 따옴표 없이 씁니다.

**예시**: 만든 뒤 바닐라라떼를 찾으면 `price` 가 `4200` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- run_cypher 에 CREATE 문 하나를 문자열로 넘긴다.

세부구현:
1. CREATE 로 Drink 레이블과 세 속성(name·price·temp)을 담은 노드를 만든다.
2. name·temp 값은 작은따옴표로, price 는 숫자로 적는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
row = run_cypher("MATCH (d:Drink {name: '바닐라라떼'}) RETURN d.price AS price")
assert len(row) == 1 and row[0]['price'] == 4200, (
    '바닐라라떼가 1개(가격 4200)가 아닙니다. 답안 셀을 두 번 실행했다면 맨 위 초기화 셀과 '
    '시드 셀부터 다시 실행한 뒤 1번을 한 번만 실행하세요'
)
print('✅ 통과!')

## 2. 새 음료를 분류에 연결하기 (CREATE 관계)
**배경**: 바닐라라떼는 **커피** 분류에 속합니다. 두 노드를 관계로 잇습니다.

**요구사항**:
- 바닐라라떼(`Drink`)와 커피(`Category`)를 각각 `MATCH` 로 찾아, 그 사이에 `IN_CATEGORY` 관계(음료→분류)를 `CREATE` 하세요.

**예시**: 연결 후 `바닐라라떼 → IN_CATEGORY → 커피` 관계가 존재합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 노드를 쉼표로 나란히 MATCH 한 뒤 그 사이에 관계를 CREATE 한다.

세부구현:
1. MATCH 로 바닐라라떼(Drink)와 커피(Category) 두 노드를 쉼표로 나란히 찾는다.
2. 이어서 그 사이에 IN_CATEGORY 관계를 CREATE 로 만든다(한 문장으로).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
hit = run_cypher(
    "MATCH (:Drink {name: '바닐라라떼'})-[:IN_CATEGORY]->(:Category {name: '커피'}) RETURN 1 AS x"
)
assert len(hit) == 1, (
    '바닐라라떼->커피 관계가 1개가 아닙니다. 답안 셀을 두 번 실행했다면 맨 위 초기화 셀과 '
    '시드 셀부터 다시 실행한 뒤 1번과 2번을 한 번씩만 실행하세요'
)
print('✅ 통과!')

## 3. 모든 음료 이름 조회하기 (MATCH 레이블)
**배경**: 메뉴판을 만들려면 음료 전체 목록이 필요합니다.

**요구사항**:
- 레이블이 `Drink` 인 노드를 **모두** 찾아 `name` 을 별칭 **`name`** 으로 RETURN 하고, 결과를 변수 **`all_drinks`** 에 담으세요.
- (**1번을 먼저 푼 상태**여야 합니다. 거기서 만든 바닐라라떼도 목록에 들어갑니다.)

**예시**: 서로 다른 음료 이름은 **8개** 입니다(11번을 먼저 풀었다면 거기서 만든 카푸치노가 더해져 9개일 수 있습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 레이블만 지정한 MATCH 로 모든 음료를 찾고 name 을 별칭으로 RETURN 한다.

세부구현:
1. 레이블 Drink 만 지정한 MATCH 로 음료를 모두 찾는다.
2. name 을 별칭 name 으로 RETURN 해 all_drinks 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert all(isinstance(r, dict) and 'name' in r for r in all_drinks), (
    'all_drinks 에는 run_cypher 결과를 그대로 담으세요(dict 의 리스트입니다). '
    '별칭이 name 인지도 확인하세요'
)
got = {r['name'] for r in all_drinks}
assert set(['녹차라떼', '딸기스무디', '바닐라라떼', '아메리카노', '아이스아메리카노', '카페라떼', '캐모마일티', '콜드브루']) <= got <= set(['녹차라떼', '딸기스무디', '바닐라라떼', '아메리카노', '아이스아메리카노', '카페라떼', '캐모마일티', '콜드브루']) | {'카푸치노'}, \
    '음료 이름 목록이 다릅니다. 레이블 Drink 만 지정해 전부 찾았는지, 별칭이 name 인지, ' \
    '그리고 1번을 먼저 풀었는지 확인하세요'
print('✅ 통과!')

## 4. 특정 음료의 가격 조회하기 (MATCH 속성 map)
**배경**: **콜드브루** 한 잔의 가격만 알고 싶습니다.

**요구사항**:
- 속성 map `{name: '콜드브루'}` 로 콜드브루 음료를 찾아 `price` 를 별칭 **`price`** 로 RETURN 하고, 결과를 변수 **`cold`** 에 담으세요.

**예시**: `cold[0]['price']` 는 **4000** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 패턴 안 속성 map 으로 콜드브루 하나만 집어 가격을 돌려받는다.

세부구현:
1. 패턴 속성 map 에 name 이 콜드브루인 조건을 담아 MATCH 로 콜드브루를 찾는다.
2. price 를 별칭 price 로 RETURN 해 cold 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(cold) == 1, (
    'cold 에 담긴 행이 한 줄이 아닙니다. 속성 map {name: \'콜드브루\'} 로 콜드브루 하나만 '
    '집었는지, run_cypher 결과를 그대로 cold 에 담았는지 확인하세요'
)
assert cold[0]['price'] == 4000, \
    '콜드브루 가격이 다릅니다. 속성 map 으로 콜드브루만 집었는지, 별칭이 price 인지 확인하세요'
print('✅ 통과!')

## 5. 4000원 이상 음료 찾기 (WHERE 비교)
**배경**: 가격대가 높은 음료만 따로 보고 싶습니다.

**요구사항**:
- `Drink` 중 `price` 가 **4000 이상**(`>=`)인 음료의 `name` 을 별칭 **`name`** 으로 RETURN 하고, 결과를 변수 **`pricey`** 에 담으세요.
- (**1번을 먼저 푼 상태**여야 합니다. 거기서 만든 바닐라라떼도 이 조건에 걸립니다.)

**예시**: 4000원 이상 음료는 **4종** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 속성 map 으로는 '이상' 을 못 거니 WHERE 에 비교 조건을 쓴다.

세부구현:
1. 레이블 Drink 로 음료를 MATCH 한다.
2. WHERE 에 price 가 4000 이상인 비교 조건을 건다.
3. name 을 별칭 name 으로 RETURN 해 pricey 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in pricey) == ['녹차라떼', '딸기스무디', '바닐라라떼', '콜드브루'], \
    '4000원 이상 음료가 다릅니다. >= 는 4000 을 포함합니다(> 가 아닙니다). 1번을 먼저 풀었는지도 확인하세요'
print('✅ 통과!')

## 6. 아이스이면서 저렴한 음료 (WHERE AND)
**배경**: **아이스** 음료 중 **4000원 이하**인 것만 고릅니다.

**요구사항**:
- `Drink` 중 `temp` 가 `'아이스'` **이고**(AND) `price` 가 **4000 이하**(`<=`)인 음료의 `name` 을 별칭 **`name`** 으로 RETURN 하고, 결과를 변수 **`cheap_ice`** 에 담으세요.

**예시**: 조건을 만족하는 음료는 **2종** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- WHERE 에 두 조건을 AND 로 잇는다(둘 다 만족).

세부구현:
1. 레이블 Drink 로 음료를 MATCH 한다.
2. WHERE 에 temp 가 아이스인 조건과 price 가 4000 이하인 조건을 AND 로 잇는다.
3. name 을 별칭 name 으로 RETURN 해 cheap_ice 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in cheap_ice) == ['아이스아메리카노', '콜드브루'], \
    '조건을 만족하는 음료가 다릅니다. 두 조건을 AND 로 이었는지, <= 가 4000 을 포함하는지 확인하세요'
print('✅ 통과!')

## 7. 핫이거나 5000원 이상인 음료 (WHERE OR)
**배경**: **따뜻한**(핫) 음료거나, 값이 비싼(**5000원 이상**) 음료를 한 번에 모아 봅니다.

**요구사항**:
- `Drink` 중 `temp` 가 `'핫'` **이거나**(OR) `price` 가 **5000 이상**(`>=`)인 음료의 `name` 을 별칭 **`name`** 으로 RETURN 하고, 결과를 변수 **`warm_or_pricey`** 에 담으세요.
- (**1번을 먼저 푼 상태**여야 합니다. 거기서 만든 바닐라라떼도 이 조건에 걸립니다.)

**예시**: 조건을 만족하는 음료는 **6종** 입니다(둘 중 **하나만** 맞아도 포함).

<details><summary>힌트</summary>

```text
접근방법:
- WHERE 에 두 조건을 OR 로 잇는다(둘 중 하나만 맞아도 통과).

세부구현:
1. 레이블 Drink 로 음료를 MATCH 한다.
2. WHERE 에 temp 가 핫인 조건과 price 가 5000 이상인 조건을 OR 로 잇는다.
3. name 을 별칭 name 으로 RETURN 해 warm_or_pricey 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in warm_or_pricey) == ['녹차라떼', '딸기스무디', '바닐라라떼', '아메리카노', '카페라떼', '캐모마일티'], \
    '조건을 만족하는 음료가 다릅니다. 두 조건을 OR 로 이었는지, 그리고 1번을 먼저 풀었는지 ' \
    '확인하세요(AND 로 이으면 결과가 줄어듭니다)'
print('✅ 통과!')

## 8. 커피 분류 음료 이름 (RETURN AS)
**배경**: **커피** 분류에 속한 음료만 이름으로 뽑습니다.

**요구사항**:
- 음료가 `IN_CATEGORY` 로 분류에 이어지는 패턴에서 분류를 `커피` 로 고정해 MATCH 하고, `d.name` 을 별칭 **`name`** 으로 RETURN 해 변수 **`coffee_names`** 에 담으세요.
- (**2번을 먼저 푼 상태**여야 합니다. 거기서 커피에 연결한 바닐라라떼도 결과에 들어갑니다.)

**예시**: 커피 분류 음료는 **5종** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 음료→분류 관계 패턴에서 분류를 커피로 고정하고 음료 이름을 돌려받는다.

세부구현:
1. MATCH 로 음료(Drink)가 IN_CATEGORY 로 분류(Category)에 이어지는 패턴을 만들되, 분류 이름을 커피로 고정한다.
2. 음료 이름을 별칭 name 으로 RETURN 해 coffee_names 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in coffee_names) == ['바닐라라떼', '아메리카노', '아이스아메리카노', '카페라떼', '콜드브루'], \
    '커피 분류 음료가 다릅니다. 화살표 방향(음료->분류)과 별칭 이름(AS name), ' \
    '그리고 2번을 먼저 풀었는지 확인하세요'
print('✅ 통과!')

## 9. 우유가 들어가는 음료 (MATCH 관계 패턴)
**배경**: 우유를 못 드시는 손님을 위해 **우유가 들어가는 음료**를 미리 추려 둡니다.

**요구사항**:
- 음료가 `HAS_INGREDIENT` 로 재료에 이어지는 패턴에서 재료를 `우유` 로 고정해 MATCH 하고, `d.name` 을 별칭 **`name`** 으로 RETURN 해 변수 **`milk_drinks`** 에 담으세요.

**예시**: 우유가 들어가는 음료는 **3종** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 8번과 같은 관계 패턴 조회다. 관계 종류와 이웃 노드의 레이블만 바뀐다.

세부구현:
1. MATCH 로 음료(Drink)가 HAS_INGREDIENT 로 재료(Ingredient)에 이어지는 패턴을 만들되, 재료 이름을 우유로 고정한다.
2. 음료 이름을 별칭 name 으로 RETURN 해 milk_drinks 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in milk_drinks) == ['녹차라떼', '딸기스무디', '카페라떼'], \
    '우유가 들어가는 음료가 다릅니다. 관계 종류(HAS_INGREDIENT)와 화살표 방향(음료->재료)을 확인하세요'
print('✅ 통과!')

## 10. 아이스 음료의 이름과 가격 (RETURN 두 별칭)
**배경**: 아이스 음료를 **이름과 가격을 함께** 정리합니다.

**요구사항**:
- `Drink` 중 `temp` 가 `'아이스'` 인 음료의 `name` 을 별칭 **`name`**, `price` 를 별칭 **`price`** 로 **함께** RETURN 하고, 결과를 변수 **`ice_menu`** 에 담으세요.

**예시**: 아이스 음료는 **3종** 이고, 각 행에 `name`·`price` 두 키가 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- RETURN 에 두 값을 쉼표로 나열하고 각각 별칭을 준다.

세부구현:
1. 레이블 Drink 로 음료를 MATCH 하고 WHERE 로 temp 가 아이스인 것만 남긴다.
2. name 과 price 를 각각 별칭 name·price 로 함께 RETURN 해 ice_menu 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted((r['name'], r['price']) for r in ice_menu) == [('딸기스무디', 5000), ('아이스아메리카노', 3500), ('콜드브루', 4000)], \
    '아이스 음료의 이름·가격이 다릅니다. RETURN 에 두 값을 나열하고 각각 별칭 name·price 를 붙였는지 확인하세요'
print('✅ 통과!')

## 11. 멱등하게 음료 추가하기 (MERGE 노드)
**배경**: 신메뉴 **카푸치노**를 넣되, 같은 코드를 **두 번 실행해도** 중복이 안 생기게 합니다.

**요구사항**:
- 실행할 Cypher 문을 **문자열 변수 `merge_q`** 에 담고, `run_cypher(merge_q)` 로 실행하세요. `Drink` 노드 `{name: '카푸치노'}` 를 **`MERGE`** 로 만드는 문이어야 합니다.
- **자가채점 셀이 `merge_q` 를 한 번 더 실행**합니다. 그래도 카푸치노가 **하나**여야 통과합니다(그래서 `CREATE` 로 쓰면 두 개가 되어 떨어집니다).

**예시**: 채점이 재실행한 뒤에도 카푸치노 노드 수는 **1** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- CREATE 대신 MERGE 를 쓰면 이미 있을 때 새로 만들지 않는다.

세부구현:
1. 레이블 Drink 에 name 이 카푸치노인 노드를 MERGE 하는 문을 문자열 변수 merge_q 에 담는다.
2. run_cypher 에 merge_q 를 넘겨 실행한다(채점 셀이 같은 문을 한 번 더 실행한다).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
up = merge_q.upper()
assert 'MERGE' in up and 'CREATE' not in up, \
    'merge_q 는 MERGE 문이어야 합니다. CREATE 로 쓰면 채점이 한 번 더 실행할 때 카푸치노가 두 개가 됩니다'
run_cypher(merge_q)   # 채점이 한 번 더 실행: 멱등이면 개수가 늘지 않는다
assert len(run_cypher("MATCH (d:Drink {name: '카푸치노'}) RETURN d")) == 1, \
    '카푸치노가 하나가 아닙니다. 답안 셀을 여러 번 실행했다면 맨 위 초기화 셀과 시드 셀부터 다시 실행하세요'
print('✅ 통과!')

## 12. 멱등하게 관계 잇기 (MERGE 관계)
**배경**: 이미 **아메리카노 → 커피** 관계가 있는데, 실수로 같은 연결을 또 실행해도 중복 관계가 생기지 않게 합니다.

**요구사항**:
- 실행할 Cypher 문을 **문자열 변수 `rel_merge_q`** 에 담고, `run_cypher(rel_merge_q)` 로 실행하세요. 아메리카노(`Drink`)와 커피(`Category`)를 `MATCH` 로 찾아 그 사이 `IN_CATEGORY` 관계(음료→분류)를 **`MERGE`** 하는 문이어야 합니다.
- **자가채점 셀이 `rel_merge_q` 를 한 번 더 실행**합니다. 그래도 그 관계가 **하나**여야 통과합니다.

**예시**: 채점이 재실행한 뒤에도 아메리카노→커피 관계 수는 **1** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 관계도 MERGE 로 만들면 이미 있을 때 중복이 안 생긴다.

세부구현:
1. 아메리카노와 커피를 MATCH 로 찾고 그 사이 IN_CATEGORY 관계를 MERGE 하는 문을 만든다.
2. 그 문을 문자열 변수 rel_merge_q 에 담고 run_cypher 로 실행한다(채점 셀이 한 번 더 실행한다).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
up = rel_merge_q.upper()
assert 'MERGE' in up and 'CREATE' not in up, \
    'rel_merge_q 는 MERGE 문이어야 합니다. CREATE 로 쓰면 채점이 한 번 더 실행할 때 관계가 두 개가 됩니다'
run_cypher(rel_merge_q)   # 채점이 한 번 더 실행: 멱등이면 관계가 늘지 않는다
assert len(run_cypher(
    "MATCH (:Drink {name: '아메리카노'})-[:IN_CATEGORY]->(:Category {name: '커피'}) RETURN 1 AS x"
)) == 1, (
    '아메리카노->커피 관계가 하나가 아닙니다. 답안 셀에 CREATE 를 썼는지 확인하고, 그랬다면 맨 위 '
    '초기화 셀과 시드 셀부터 다시 실행하세요'
)
print('✅ 통과!')

## 13. 재료의 양을 관계에 담기 (CREATE 관계 속성)
**배경**: 바닐라라떼 한 잔에 **원두 18g** 이 들어갑니다. 이 `18` 은 음료의 값도 재료의 값도 아닙니다. 같은 원두라도 음료마다 양이 다르니까요. **그 연결 하나의 값**이므로 관계에 담습니다. (1번을 먼저 풀어 바닐라라떼가 있어야 합니다.)

**요구사항**:
- 바닐라라떼(`Drink`)와 원두(`Ingredient`)를 각각 `MATCH` 로 찾아, 그 사이에 `HAS_INGREDIENT` 관계(음료→재료)를 `CREATE` 하되 관계에 속성 **`{amount: 18}`** 을 붙이세요.
- 관계에 변수 `r` 을 붙이고 **같은 문장 끝에 `RETURN`** 을 이어 써 `r.amount` 를 별칭 **`amount`** 로 돌려받으세요. 그 문장을 **문자열 변수 `bean_q`** 에 담고 **`bean_amount = run_cypher(bean_q)`** 로 실행해 출력합니다. **자가채점 셀이 `bean_q` 를 읽어** 만드는 문장 안에 `RETURN` 이 있는지 확인합니다(따로 `MATCH` 해서 받아 오면 걸립니다).
- **이 답안 셀도 한 번만 실행하세요**(`CREATE` 라 두 번 실행하면 관계가 두 개가 됩니다).

**예시**: `bean_amount` 가 `[{'amount': 18}]` 한 줄이면 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 관계 속성은 노드 속성과 똑같이 {키: 값} 인데, 자리만 대괄호 안이다.

세부구현:
1. 바닐라라떼와 원두를 쉼표로 나란히 MATCH 한다.
2. 그 사이에 관계를 CREATE 하되 대괄호 안에 변수 r 과 {amount: 18} 을 함께 적는다.
3. 같은 문장 끝에 RETURN 을 이어 써 r.amount 를 별칭 amount 로 돌려받는다.
4. 그 문장을 문자열 변수 bean_q 에 담고 run_cypher 에 넘겨 bean_amount 에 담아 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
up = bean_q.upper()
assert 'CREATE' in up and 'RETURN' in up, (
    'bean_q 한 문장 안에 CREATE 와 RETURN 이 함께 있어야 합니다. 관계를 만든 뒤 따로 MATCH 해서 '
    '받아 오는 것이 아니라, 만드는 그 문장이 값을 돌려주게 하세요'
)
# 1) 그래프에 관계와 속성이 제대로 들어갔는지
rows = run_cypher(
    "MATCH (:Drink {name: '바닐라라떼'})-[r:HAS_INGREDIENT]->(:Ingredient {name: '원두'}) "
    "RETURN r.amount AS amount"
)
assert len(rows) == 1, (
    '바닐라라떼->원두 관계가 하나가 아닙니다. 답안 셀을 두 번 실행했다면 맨 위 초기화 셀과 '
    '시드 셀부터 다시 실행한 뒤 1번부터 한 번씩만 푸세요'
)
assert rows[0]['amount'] == 18, \
    '관계 속성 amount 값이 다릅니다. 대괄호 안에 {amount: 18} 을 적었는지 확인하세요'
# 2) 만드는 그 문장에서 RETURN 으로 돌려받았는지(bean_amount 가 그 증거다)
assert bean_amount == [{'amount': 18}], (
    'bean_amount 가 다릅니다. CREATE 문 끝에 RETURN r.amount AS amount 를 이어 써 그 결과를 '
    'bean_amount 에 담으세요(다시 MATCH 해서 받은 값이 아니라 만드는 문장이 돌려준 값입니다)'
)
print('✅ 통과!')

## 14. 재료 관계만 지우기 (DELETE)
**배경**: 레시피가 바뀌어 바닐라라떼의 **원두 재료 표기를 빼기로** 했습니다. 음료 자체는 남기고 **연결만** 끊습니다. (13번을 먼저 풀어야 합니다.)

**요구사항**:
- 바닐라라떼에서 나가는 `HAS_INGREDIENT` 관계를 **관계 변수 `r` 을 붙여 `MATCH`** 한 뒤 **`DELETE r`** 로 지우세요.
- **바닐라라떼 노드는 지우면 안 됩니다.** `DETACH DELETE` 를 쓰면 노드까지 사라져 떨어집니다.
- 지운 뒤 바닐라라떼의 남은 `HAS_INGREDIENT` 관계 수를 `len(run_cypher(...))` 로 세어 **출력해 눈으로 확인**하세요(이 출력 자체는 채점하지 않습니다. 채점은 그래프 상태를 봅니다).

**예시**: 남은 재료 관계는 **0** 개이고, 바닐라라떼 노드는 그대로 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 지울 대상이 관계이므로 관계 쪽에 변수를 붙인다.

세부구현:
1. (:Drink {name: '바닐라라떼'})-[r:HAS_INGREDIENT]->() 패턴을 MATCH 한다.
2. 이어서 DELETE r 을 적는다(노드가 아니라 r 을 지운다).
3. 같은 패턴을 다시 MATCH 해 len() 으로 세어 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(run_cypher(
    "MATCH (:Drink {name: '바닐라라떼'})-[r:HAS_INGREDIENT]->() RETURN r"
)) == 0, (
    '바닐라라떼의 재료 관계가 아직 남아 있습니다. 관계에 변수 r 을 붙여 MATCH 한 뒤 '
    'DELETE r 을 적었는지 확인하세요'
)
assert len(run_cypher("MATCH (d:Drink {name: '바닐라라떼'}) RETURN d")) == 1, (
    '바닐라라떼 노드까지 사라졌습니다. 이 문제는 관계만 지우는 문제입니다. '
    'DETACH DELETE 가 아니라 DELETE r 을 쓰세요(다시 풀려면 맨 위 초기화 셀부터 실행)'
)
print('✅ 통과!')

## 15. 단종된 음료 지우기 (DETACH DELETE)
**배경**: 바닐라라떼가 결국 **단종**됐습니다. 노드를 그래프에서 뺍니다. 2번에서 이은 **분류 관계(`IN_CATEGORY`)가 아직 붙어 있으니** 그냥 `DELETE` 로는 거부됩니다.

**요구사항**:
- 바닐라라떼(`Drink`)를 `MATCH` 로 찾아 **`DETACH DELETE`** 로 지우세요.
- 지운 뒤 남은 `Drink` 노드 수를 `len(run_cypher(...))` 로 세어 **출력해 눈으로 확인**하세요(이 출력 자체는 채점하지 않습니다).
- **13번까지 다 푼 뒤에 푸세요.** 앞 문제들이 만든 것을 지우므로, 먼저 풀면 앞 문제의 채점이 떨어집니다. 16번부터는 바닐라라떼를 쓰지 않으니 이어서 그대로 풀면 됩니다.

**예시**: 남은 음료는 **8종** 입니다(시드 7종 + 11번의 카푸치노, 바닐라라떼는 빠짐).

<details><summary>힌트</summary>

```text
접근방법:
- 관계가 붙은 노드는 DETACH 를 붙여야 지워진다.

세부구현:
1. MATCH 로 바닐라라떼(Drink)를 찾는다.
2. 이어서 DETACH DELETE 를 적는다(붙은 관계까지 함께 떼어 내고 지운다).
3. Drink 노드를 전부 MATCH 해 len() 으로 세어 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(run_cypher("MATCH (d:Drink {name: '바닐라라떼'}) RETURN d")) == 0, \
    '바닐라라떼가 아직 남아 있습니다. 관계가 붙어 있으면 DETACH 를 붙여야 지워집니다'
assert len(run_cypher("MATCH (d:Drink) RETURN d")) == 8, (
    '남은 음료 수가 다릅니다. 1·11번을 풀어 두 음료를 추가한 상태에서 바닐라라떼만 빠져야 합니다. '
    '어긋났다면 맨 위 초기화 셀과 시드 셀부터 다시 실행한 뒤 1번부터 순서대로 푸세요'
)
print('✅ 통과!')

## 16. 값이 비어 있는 음료 찾기 (IS NULL)
**배경**: 11번에서 `MERGE` 로 만든 **카푸치노**는 이름만 적어 만들었습니다. 그래서 가격이 아예 없습니다. 채워 넣어야 할 음료를 먼저 찾아 봅니다. (**11번을 먼저 푼 상태**여야 합니다.)

**요구사항**:
- `Drink` 중 `price` 속성이 **아예 없는** 음료의 `name` 을 별칭 **`name`** 으로 RETURN 하고, 결과를 변수 **`no_price`** 에 담으세요.
- 값이 없는 것은 `=` 비교로 걸러지지 않습니다(없는 값은 어떤 값과도 같지 않습니다). `WHERE` 에 **`IS NULL`** 을 쓰세요.

**예시**: 가격이 비어 있는 음료는 **1종** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 없는 값은 비교로 못 걸러낸다. '값이 없다' 를 직접 묻는 조건을 쓴다.

세부구현:
1. 레이블 Drink 로 음료를 MATCH 한다.
2. WHERE 에 price 가 비어 있다는 조건을 IS NULL 로 적는다.
3. name 을 별칭 name 으로 RETURN 해 no_price 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert all(isinstance(r, dict) and 'name' in r for r in no_price), (
    'no_price 에는 run_cypher 결과를 그대로 담으세요(dict 의 리스트입니다). '
    '별칭이 name 인지도 확인하세요'
)
assert sorted(r['name'] for r in no_price) == ['카푸치노'], (
    '가격이 비어 있는 음료가 다릅니다. 11번(MERGE 로 카푸치노 만들기)을 먼저 풀었는지, '
    'WHERE 조건을 IS NULL 로 적었는지 확인하세요'
)
print('✅ 통과!')

## 17. 빠진 값 채우고 표시 붙이기 (SET)
**배경**: 16번에서 찾은 **카푸치노**에 가격과 온도를 채우고, 신메뉴라는 표시도 함께 답니다. 이미 있는 노드를 **고치는** 일이라 `CREATE` 가 아니라 `SET` 입니다. (**16번 다음에 푸세요**.)

**요구사항**:
- 카푸치노(`Drink`)를 `MATCH` 로 찾아 `SET` 으로 `price` 를 **4300**, `temp` 를 **`'핫'`** 으로 넣으세요(쉼표로 나열하면 한 문장에서 둘 다 넣을 수 있습니다).
- 같은 `SET` 에 레이블 **`New`** 도 함께 붙이세요. 레이블은 속성과 달리 `=` 로 값을 주는 것이 아니라 **콜론을 붙여 이름만**(`:New`) 적습니다.
- 바뀐 값을 다시 조회해 **출력해 눈으로 확인**하세요(이 출력 자체는 채점하지 않습니다. 채점은 그래프 상태를 봅니다).
- 이어서 **`Drink` 이면서 `New` 인** 노드를 **두 레이블을 나란히 적은 패턴**(`(d:Drink:New)`)으로 찾아 `name` 을 별칭 **`name`** 으로 RETURN 하고, 그 결과를 변수 **`new_drinks`** 에 담아 정렬해 출력하세요. 레이블을 나란히 적은 것은 **AND** 라 둘 다 가진 노드만 걸립니다(교안_01 1-3).
- **이 문제부터 그래프의 값이 바뀝니다.** 5·7·10번 같은 앞 문제의 채점 셀을 지금 다시 실행하면 결과가 달라져 떨어집니다(순서대로 한 번씩 풀었다면 다시 실행할 필요가 없습니다).

**예시**: 카푸치노의 `price` 는 **4300**, `temp` 는 **'핫'** 이 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 있는 노드를 고치는 자리다. MATCH 로 집어 SET 을 이어 쓴다.
- 속성과 레이블은 같은 SET 안에 쉼표로 나란히 적을 수 있다.

세부구현:
1. MATCH 로 카푸치노(Drink)를 찾는다.
2. 이어서 SET 을 쓰고 price 와 temp 를 쉼표로 나란히 적는다.
3. 같은 SET 뒤에 레이블 New 를 콜론 형태로 하나 더 적는다.
4. 카푸치노를 다시 MATCH 해 price·temp 를 돌려받아 출력한다.
5. MATCH (d:Drink:New) 로 두 레이블을 다 가진 노드를 찾아 new_drinks 에 담고 정렬 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
row = run_cypher("MATCH (d:Drink {name: '카푸치노'}) RETURN d.price AS price, d.temp AS temp")
assert len(row) == 1, (
    '카푸치노가 하나가 아닙니다. 11번(MERGE 로 카푸치노 만들기)을 먼저 풀었는지 확인하세요'
)
assert row[0]['price'] == 4300 and row[0]['temp'] == '핫', (
    '카푸치노의 price·temp 가 다릅니다. SET 에 두 속성을 쉼표로 나란히 적었는지, '
    '숫자에는 따옴표를 붙이지 않았는지 확인하세요'
)
assert len(run_cypher("MATCH (d:Drink:New) RETURN d")) == 1, (
    '레이블 New 가 카푸치노에 붙어 있지 않습니다. 같은 SET 안에 콜론을 붙인 레이블 이름을 '
    '쉼표로 하나 더 적으세요'
)
assert sorted(r['name'] for r in new_drinks) == ['카푸치노'], (
    'new_drinks 가 다릅니다. 두 레이블을 나란히 적은 패턴 (d:Drink:New) 으로 찾았는지, '
    '별칭이 name 인지 확인하세요. 레이블 나열은 AND 라 둘 다 가진 노드만 걸립니다'
)
print('✅ 통과!')

## 18. 표기 지우기 (REMOVE)
**배경**: **캐모마일티**는 핫·아이스 둘 다 팔기로 해서 `temp` 표기를 아예 뺍니다. 그리고 17번에서 카푸치노에 붙인 신메뉴 표시(`New` 레이블)도 이제 뗍니다. (**17번을 먼저 풀어야 합니다**.)

**요구사항**: 문장 **두 개**를 각각 `run_cypher` 로 실행하세요.
- 캐모마일티(`Drink`)를 `MATCH` 로 찾아 **`REMOVE`** 로 속성 `temp` 를 지우세요.
- 카푸치노(`Drink`)를 `MATCH` 로 찾아 **`REMOVE`** 로 레이블 **`New`** 를 떼세요(레이블은 17번에서 붙일 때처럼 콜론을 붙여 적습니다).
- 캐모마일티의 `temp` 를 다시 조회해 **출력해 눈으로 확인**하세요(이 출력 자체는 채점하지 않습니다).

**예시**: 캐모마일티의 `temp` 는 **비어 있게**(`None`) 되고, `New` 레이블이 붙은 노드는 하나도 남지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- SET 이 값을 얹는 자리라면 REMOVE 는 그 자리를 통째로 없애는 짝이다.
- 속성과 레이블은 적는 모양만 다르다(하나는 점, 하나는 콜론).

세부구현:
1. MATCH 로 캐모마일티를 찾고 이어서 REMOVE 로 temp 속성을 지운다.
2. MATCH 로 카푸치노를 찾고 이어서 REMOVE 로 New 레이블을 뗀다.
3. 캐모마일티를 다시 MATCH 해 temp 를 돌려받아 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
row = run_cypher("MATCH (d:Drink {name: '캐모마일티'}) RETURN d.temp AS temp")
assert len(row) == 1, '캐모마일티를 찾지 못했습니다. 맨 위 초기화 셀과 시드 셀부터 다시 실행하세요'
assert row[0]['temp'] is None, (
    '캐모마일티의 temp 가 아직 남아 있습니다. REMOVE 로 속성을 지웠는지 확인하세요'
)
assert len(run_cypher("MATCH (d:Drink:New) RETURN d")) == 0, (
    'New 레이블이 아직 붙어 있습니다. 레이블은 콜론을 붙여(REMOVE d:New 처럼) 뗍니다'
)
print('✅ 통과!')

## 19. 아이스 음료가 있는 분류 (DISTINCT)
**배경**: 아이스 음료를 파는 **분류**가 어디어디인지 봅니다. 한 분류에 아이스 음료가 여러 개면 같은 분류 이름이 그 수만큼 여러 줄로 나옵니다. 필요한 것은 **종류**뿐입니다.

**요구사항**:
- 음료가 `IN_CATEGORY` 로 분류에 이어지는 패턴을 MATCH 하고, `WHERE` 로 음료의 `temp` 가 `'아이스'` 인 것만 남긴 뒤, **분류 이름**을 별칭 **`name`** 으로 RETURN 해 변수 **`ice_cats`** 에 담으세요.
- 같은 분류가 여러 줄 나오지 않도록 **`DISTINCT`** 로 중복을 없애세요(`RETURN DISTINCT ...` 처럼 RETURN 바로 뒤에 붙입니다).
- 실행할 Cypher 문을 **문자열 변수 `ice_cats_q`** 에 담고 **`ice_cats = run_cypher(ice_cats_q)`** 로 실행하세요. **자가채점 셀이 `ice_cats_q` 를 한 번 더 실행**해, 중복을 **Cypher 가** 없앴는지 확인합니다. 파이썬 집합으로 지우면 여기서 걸립니다.

**예시**: `ice_cats` 는 **2줄** 입니다(중복을 없애지 않으면 3줄이 나옵니다).

<details><summary>힌트</summary>

```text
접근방법:
- 음료 쪽에 조건을 걸고 돌려받는 것은 분류 이름이다. 음료가 여럿이면 같은 분류가 겹쳐 나온다.
- 겹치는 줄은 RETURN 자리에서 없앤다.

세부구현:
1. MATCH 로 음료(Drink)가 IN_CATEGORY 로 분류(Category)에 이어지는 패턴을 만든다.
2. WHERE 로 음료의 temp 가 아이스인 것만 남긴다.
3. RETURN 바로 뒤에 DISTINCT 를 붙이고 분류 이름을 별칭 name 으로 돌려받는다.
4. 그 문장을 문자열 변수 ice_cats_q 에 담고 run_cypher 에 넘겨 결과를 ice_cats 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
up = ice_cats_q.upper()
assert 'DISTINCT' in up, (
    'ice_cats_q 에 DISTINCT 가 없습니다. 중복을 없애는 일은 파이썬이 아니라 Cypher 가 해야 합니다'
)
again = run_cypher(ice_cats_q)   # 채점이 같은 문을 한 번 더 실행: 그 문장만으로 접혀야 한다
assert len(again) == 2, (
    'ice_cats_q 가 돌려주는 줄이 2줄이 아닙니다. RETURN 뒤에 DISTINCT 를 붙여 '
    '중복을 없앴는지 확인하세요(없애지 않으면 3줄이 나옵니다)'
)
assert sorted(r['name'] for r in again) == ['스무디', '커피'], (
    'ice_cats_q 가 돌려주는 분류 이름이 다릅니다. 돌려받는 것이 음료 이름이 아니라 분류 이름인지, '
    '별칭이 name 인지, WHERE 조건이 아이스인지 확인하세요'
)
assert [r['name'] for r in ice_cats] == [r['name'] for r in again], (
    'ice_cats 에는 run_cypher(ice_cats_q) 의 결과를 그대로 담으세요(파이썬에서 다시 손대지 마세요)'
)
print('✅ 통과!')

## 20. 가장 비싼 음료 3종 (ORDER BY 와 LIMIT)
**배경**: 메뉴판 맨 위에 올릴 **가장 비싼 음료 3종**을 비싼 순서 그대로 뽑습니다. (**17번에서 카푸치노 가격을 채운 뒤**여야 합니다.)

**요구사항**:
- `Drink` 의 `name` 을 별칭 **`name`**, `price` 를 별칭 **`price`** 로 **함께** RETURN 하되, **`ORDER BY`** 로 가격이 **큰 값부터**(내림차순 `DESC`) 줄 세우고, **`LIMIT`** 으로 앞에서 **3줄**만 남기세요.
- 실행할 Cypher 문을 **문자열 변수 `top3_q`** 에 담고, **`top3 = run_cypher(top3_q)`** 로 실행하세요. **자가채점 셀이 `top3_q` 를 한 번 더 실행**해, 그 문장 하나만으로 답이 나오는지 확인합니다.
- 그래서 이 문제는 **파이썬에서 정렬하거나 잘라내면 통과하지 못합니다.** 줄 세우기와 끊기를 모두 Cypher 안에서 하세요(순서까지 채점합니다).

**예시**: `top3` 는 3줄이고, 첫 줄이 가장 비싼 음료(**5000원**)입니다.

<details><summary>힌트</summary>

```text
접근방법:
- '가장 비싼 몇 개' 는 두 걸음이다. 먼저 줄을 세우고, 그다음 앞에서 끊는다.
- ORDER BY 에는 RETURN 에서 붙인 별칭을 그대로 써도 된다.

세부구현:
1. 레이블 Drink 로 음료를 MATCH 하고 name·price 를 별칭과 함께 RETURN 하는 문을 만든다.
2. RETURN 뒤에 ORDER BY 를 이어 쓰고 가격 기준으로 내림차순 정렬한다(DESC).
3. 그 뒤에 LIMIT 으로 앞 3줄만 남긴다.
4. 그 문장을 문자열 변수 top3_q 에 담고 run_cypher 에 넘겨 결과를 top3 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
up = top3_q.upper()
assert 'ORDER BY' in up, (
    'top3_q 에 ORDER BY 가 없습니다. 줄 세우기는 파이썬이 아니라 Cypher 가 해야 합니다'
)
assert 'LIMIT' in up, (
    'top3_q 에 LIMIT 이 없습니다. 앞에서 3줄만 남기는 것도 Cypher 안에서 하세요'
)
again = run_cypher(top3_q)   # 채점이 같은 문을 한 번 더 실행: 그 문장만으로 답이 나와야 한다
assert len(again) == 3, (
    'top3_q 가 돌려주는 줄이 3줄이 아닙니다. LIMIT 으로 앞에서 3줄만 남겼는지 확인하세요'
)
assert all(isinstance(r, dict) and {'name', 'price'} <= set(r) for r in again), (
    '각 행에 별칭 name 과 price 가 함께 있어야 합니다. RETURN 에 두 값을 나열했는지 확인하세요'
)
assert [(r['name'], r['price']) for r in again] == [('딸기스무디', 5000), ('녹차라떼', 4500), ('카푸치노', 4300)], (
    '음료 3종이나 그 순서가 다릅니다. ORDER BY 에 DESC 를 붙여 큰 값부터 줄 세웠는지, '
    '17번에서 카푸치노 가격을 채웠는지 확인하세요(가격이 비어 있는 음료가 있으면 맨 앞으로 옵니다)'
)
assert [(r['name'], r['price']) for r in top3] == [('딸기스무디', 5000), ('녹차라떼', 4500), ('카푸치노', 4300)], (
    'top3 에는 run_cypher(top3_q) 의 결과를 그대로 담으세요(파이썬에서 다시 정렬하거나 '
    '잘라내지 마세요)'
)
print('✅ 통과!')

## 21. 아이스가 아닌 음료, 그리고 빠진 한 잔 (NOT 과 IS NULL)
**배경**: 따뜻한 음료만 추리려고 **아이스가 아닌** 음료를 `NOT` 으로 찾습니다. 그런데 18번에서 `temp` 표기를 지운 **캐모마일티**가 결과에 없습니다. 지운 적도 없는데 왜 빠졌을까요. (**18번을 먼저 푼 상태**여야 합니다.)

**요구사항**: 문장 **두 개**를 각각 실행합니다.
- `Drink` 를 MATCH 하고 `WHERE` 에 **`NOT`** 을 써서 `temp` 가 `'아이스'` **가 아닌** 음료를 남기세요. `name` 을 별칭 **`name`** 으로 RETURN 해 변수 **`not_ice`** 에 담고 출력합니다.
- 이어서 `temp` 속성이 **아예 없는** 음료를 **`IS NULL`** 로 찾아 같은 별칭 **`name`** 으로 RETURN 해 변수 **`no_temp`** 에 담고 출력하세요.

**예시**: 앞 결과는 **4종**, 뒤 결과는 **1종** 이고 **서로 겹치지 않습니다.** 값이 아예 없는 음료는 `= '아이스'` 에도 `NOT = '아이스'` 에도 걸리지 않기 때문입니다. 그래서 그런 음료를 찾으려면 `IS NULL` 이 따로 필요합니다.

<details><summary>힌트</summary>

```text
접근방법:
- NOT 은 뒤에 오는 조건을 통째로 뒤집는다. '아이스가 아니다' 를 그대로 옮겨 적으면 된다.
- 값이 없는 것은 어떤 값과 비교해도 참이 되지 않는다. 그래서 NOT 으로도 못 찾는다.

세부구현:
1. 레이블 Drink 로 음료를 MATCH 하고, WHERE 에 NOT 을 붙여 temp 가 아이스인 조건을 뒤집는다.
2. name 을 별칭 name 으로 RETURN 해 not_ice 에 담고 출력한다.
3. 같은 레이블을 다시 MATCH 하고 WHERE 에 temp 가 IS NULL 인 조건을 걸어 no_temp 에 담고 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in not_ice) == ['녹차라떼', '아메리카노', '카페라떼', '카푸치노'], (
    '아이스가 아닌 음료가 다릅니다. WHERE 에 NOT 을 붙여 temp 조건을 뒤집었는지 확인하세요. '
    '18번까지 순서대로 풀었는지도 보세요(캐모마일티의 temp 가 지워져 있어야 합니다)'
)
assert sorted(r['name'] for r in no_temp) == ['캐모마일티'], (
    'temp 가 없는 음료가 다릅니다. IS NULL 로 찾았는지, 18번에서 캐모마일티의 temp 를 '
    '지웠는지 확인하세요'
)
assert not (set(r['name'] for r in not_ice) & set(r['name'] for r in no_temp)), \
    '두 결과가 겹칩니다. NOT 으로 찾은 목록에는 값이 아예 없는 음료가 들어가면 안 됩니다'
print('✅ 통과!')

## 22. 새 음료와 새 분류를 한 문장으로 (CREATE 한 문장)
**배경**: 신메뉴 **말차라떼**를 새로 만든 분류 **논커피**에 넣습니다. 2번에서는 커피 분류가 **이미 있었기 때문에** `MATCH` 로 찾아 이었죠. 이번에는 음료도 분류도 아직 없습니다. **없는 것은 `MATCH` 로 잡을 수 없으니** 화살표 양 끝에 노드 패턴을 그대로 그려 한 문장에 다 만듭니다.

**요구사항**:
- 음료 노드에 변수 `d`, 분류 노드에 변수 `c` 를 붙여 **한 문장의 `CREATE`** 로 음료·분류·관계를 함께 만드세요. 음료는 `{name: '말차라떼', price: 5200, temp: '핫'}`, 분류는 `{name: '논커피'}`, 관계는 `IN_CATEGORY`(음료→분류)입니다.
- **같은 문장 끝에 `RETURN`** 을 이어 써 `d.name` 을 별칭 **`name`**, `c.name` 을 별칭 **`category`** 로 돌려받으세요.
- 그 문장을 **문자열 변수 `menu_q`** 에 담고 **`new_menu = run_cypher(menu_q)`** 로 실행한 뒤 출력하세요. **자가채점 셀이 `menu_q` 를 읽어** 정말 한 문장인지(`MATCH` 없이 `CREATE` 하나로 끝났는지) 확인합니다.
- **이 답안 셀도 한 번만 실행하세요**(`CREATE` 라 두 번 실행하면 말차라떼도 논커피도 두 개가 됩니다).

**예시**: `new_menu` 가 `[{'name': '말차라떼', 'category': '논커피'}]` 한 줄이면 됩니다. 두 값이 **한 문장에서 함께** 나온다는 것이 곧 노드 둘을 한 번에 만들었다는 뜻입니다.

**주의**: 이 형태는 **양쪽이 모두 새것일 때만** 씁니다. 한쪽이 이미 있는데 이렇게 쓰면 이름만 같은 노드가 하나 더 생겨 조회 결과가 둘로 갈라집니다(그래서 2번은 `MATCH` 로 찾아 이었습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 관계를 그으려면 양 끝이 필요한데, 아직 없는 노드는 찾을 수가 없다.
- 화살표 양 끝에 노드 패턴을 통째로 적으면 그 자리에서 노드도 함께 만들어진다.
- 두 노드에 변수를 붙여야 같은 문장의 RETURN 이 둘 다 가리킬 수 있다.

세부구현:
1. CREATE 뒤에 음료 노드 패턴을 변수 d 와 함께 적고, 이어서 -[:IN_CATEGORY]-> 를 그린다.
2. 화살표 오른쪽에 분류 노드 패턴을 변수 c 와 함께 적는다(MATCH 는 쓰지 않는다).
3. 같은 문장 끝에 RETURN d.name AS name, c.name AS category 를 이어 쓴다.
4. 그 문장 전체를 문자열 변수 menu_q 에 담고 run_cypher 에 넘겨 결과를 new_menu 에 담아 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
up = menu_q.upper()
assert 'MATCH' not in up, (
    'menu_q 에 MATCH 가 있습니다. 아직 없는 노드는 MATCH 로 잡을 수 없습니다. MATCH 없이 화살표 '
    '양 끝에 노드 패턴을 그대로 그려 한 문장으로 만드세요'
)
assert up.count('CREATE') == 1, (
    'CREATE 를 한 번만 쓰세요. 이 문제는 문장 하나가 노드 2개와 관계 1개를 함께 만드는 형태를 '
    '익히는 자리입니다(따로따로 만들면 그 형태를 쓰지 않은 것입니다)'
)
assert new_menu == [{'name': '말차라떼', 'category': '논커피'}], (
    'new_menu 가 다릅니다. 한 문장의 CREATE 안에서 두 노드에 변수 d·c 를 붙이고, 같은 문장 끝에 '
    'RETURN d.name AS name, c.name AS category 를 이어 썼는지 확인하세요(두 값이 한 줄에 함께 나와야 합니다)'
)
assert len(run_cypher("MATCH (d:Drink {name: '말차라떼'}) RETURN d")) == 1, (
    '말차라떼가 하나가 아닙니다. 답안 셀을 두 번 실행했다면 맨 위 초기화 셀과 시드 셀부터 다시 '
    '실행한 뒤 1번부터 한 번씩만 푸세요'
)
assert len(run_cypher("MATCH (c:Category {name: '논커피'}) RETURN c")) == 1, (
    '논커피 분류가 하나가 아닙니다. 분류 노드도 이 한 문장에서 딱 한 번만 만들어져야 합니다'
)
assert len(run_cypher(
    "MATCH (:Drink {name: '말차라떼'})-[:IN_CATEGORY]->(:Category {name: '논커피'}) RETURN 1 AS x"
)) == 1, (
    '말차라떼->논커피 관계가 하나가 아닙니다. 화살표를 두 노드 패턴 사이에 한 번만 그렸는지 확인하세요'
)
print('✅ 통과!')

## 23. 자료형을 갖춰 넣고 날짜로 찾기 (자료형·`date()`)
**배경**: 지금까지 넣은 값은 **문자열과 정수** 둘뿐이었습니다. 이번 신메뉴 **제주말차**에는 별점(실수)·시그니처 여부(불리언)·출시일(날짜)까지 함께 담습니다. 특히 날짜는 **따옴표로 적으면 그냥 글자**가 되어 나중에 크기 비교가 되지 않습니다.

**요구사항**: 문장 **두 개**를 각각 실행합니다.
- `Drink` 노드를 `CREATE` 하되 속성을 이렇게 적으세요. `name: '제주말차'`, `price: 5800`, `temp: '핫'`, `rating: 4.7`, `signature: true`, `launched: date('2026-03-02')`. **불리언은 소문자 `true`**(파이썬의 `True` 가 아닙니다)이고, **날짜는 `date('2026-03-02')`** 로 적습니다(따옴표만 씌우면 안 됩니다).
- 노드에 변수 `d` 를 붙이고 **같은 문장 끝에 `RETURN`** 을 이어 써 `price`·`rating`·`signature`·`launched` 를 각각 **같은 이름의 별칭**으로 돌려받아 변수 **`new_drink`** 에 담고 출력하세요.
- 이어서 **출시일이 `2026-01-01` 이후**인 음료를 `WHERE` 와 `>=` 로 찾아 `name` 을 별칭 **`name`** 으로 RETURN 하고, 그 결과를 변수 **`recent`** 에 담아 정렬해 출력하세요.
- **이 답안 셀은 한 번만 실행하세요**(`CREATE` 라 두 번 실행하면 제주말차가 두 개가 됩니다).

**예시**: `new_drink` 는 한 줄이고, 파이썬에서 `price` 는 정수, `rating` 은 실수, `signature` 는 참·거짓, `launched` 는 날짜 값으로 옵니다. 뒤 결과 `recent` 는 **1종** 입니다. 다른 음료에는 `launched` 가 아예 없어서, 값이 없으면 비교가 성립하지 않아 조건에서 빠지기 때문입니다(16·21번에서 본 그 규칙입니다).

<details><summary>힌트</summary>

```text
접근방법:
- 자료형마다 적는 법이 다르다. 문자열만 따옴표로 감싸고 숫자·불리언은 그대로 적는다.
- 날짜는 date('...') 로 만든다. 따옴표만 씌운 '2026-03-02' 은 날짜가 아니라 글자다.
- 날짜끼리는 숫자처럼 >= 로 앞뒤를 견줄 수 있다.

세부구현:
1. CREATE 로 Drink 노드를 만들되 여섯 속성을 지문에 적힌 표기 그대로 적는다(변수 d).
2. 같은 문장 끝에 RETURN 을 이어 써 네 값을 같은 이름의 별칭으로 돌려받아 new_drink 에 담고 출력한다.
3. Drink 를 MATCH 하고 WHERE 에 launched 가 date('2026-01-01') 이상인 조건을 건다.
4. name 을 별칭 name 으로 RETURN 해 recent 에 담고 정렬해 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
row = new_drink[0]
assert row['price'] == 5800 and isinstance(row['price'], int) and not isinstance(row['price'], bool), (
    'price 는 따옴표 없이 정수로 적어야 합니다(5800). 따옴표를 씌우면 문자열이 됩니다'
)
assert row['rating'] == 4.7 and isinstance(row['rating'], float), (
    'rating 은 소수점을 그대로 적어 실수로 넣으세요(4.7). Cypher 는 정수와 실수를 구분합니다'
)
assert row['signature'] is True, (
    'signature 는 소문자 true 로 적어야 불리언이 됩니다. 파이썬처럼 True 라고 쓰거나 '
    "'true' 로 따옴표를 씌우면 안 됩니다"
)
assert type(row['launched']).__name__ == 'Date', (
    'launched 가 날짜가 아닙니다. date(\'2026-03-02\') 로 적었는지 확인하세요. '
    '따옴표만 씌우면 그냥 글자(str)가 됩니다'
)
assert str(row['launched']) == '2026-03-02', 'launched 날짜가 다릅니다'
assert sorted(r['name'] for r in recent) == ['제주말차'], (
    '출시일로 찾은 음료가 다릅니다. WHERE 에 launched >= date(\'2026-01-01\') 조건을 걸었는지 '
    '확인하세요. 다른 음료에는 launched 가 없어 걸리지 않는 것이 정상입니다'
)
print('✅ 통과!')

---
수고했어요! LV1 에서 CREATE(노드·관계·관계 속성)·MATCH·WHERE(비교·AND·OR·NOT·IS NULL)·RETURN(별칭·DISTINCT·ORDER BY·LIMIT)·SET·REMOVE·MERGE(노드·관계)·DELETE(관계·노드)를 **하나씩** 익혔습니다. 마지막에는 양쪽이 모두 새것일 때 쓰는 **한 문장 생성**과 **속성의 자료형·날짜**까지 다뤘습니다. LV2 에서는 이것들을 **조합**해 여러 노드를 잇는 패턴 질의를 다룹니다.